<a href="https://colab.research.google.com/github/Shiveshrane/Research_paper_implementations/blob/main/BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import tensorflow as tf
import numpy as np

In [5]:
class Positional_Embedding(tf.keras.layers.Layer):
  def __init__(self,d_model, max_len=512):
    super(Positional_Embedding, self).__init__()
    self.d_model=d_model
    self.max_len=max_len
    self.depth=d_model//2
    self.pos_encodings=self.get_pos_encodings(max_len, d_model)
    self.pos_encodings=tf.Variable(self.pos_encodings, trainable=False)

  def get_pos_encodings(self, max_len, d_model):
    positions=tf.range(self.max_len)[:, tf.newaxis]
    depth=tf.range(self.depth)[tf.newaxis, :]/self.depth # 2i/d_model
    angles=tf.cast(positions, tf.float32)*(1/tf.pow(10000, tf.cast(depth, tf.float32)))
    pos_encodings=tf.concat([tf.sin(angles), tf.cos(angles)], axis=-1)
    return pos_encodings

  def call(self, X, training=False):
    seq_len=tf.shape(X)[1]
    pos_enc=self.pos_encodings[:seq_len, :] ## Making it dynamic
    pos_enc=tf.expand_dims(pos_enc, axis=0)
    X=X+pos_enc
    return X


In [27]:
class Embedding(tf.keras.layers.Layer):
  def __init__(self, vocab_size, d_model):
    super(Embedding, self).__init__()
    self.d_model=d_model
    self.vocab_size=vocab_size
    self.embedding=tf.keras.layers.Embedding(vocab_size, d_model)
    self.Pos_embed=Positional_Embedding(d_model)
    self.segment_embed=tf.keras.layers.Embedding(2, d_model)

  def call(self, X, segment_id=None, training=False):
    X=self.embedding(X)
    X=X+self.Pos_embed(X, training=training)
    if segment_id is not None:
      seg_embed=self.segment_embed(segment_id)
      X=X+seg_embed
    return X

In [6]:
#test embedding
vocab_size=100
d_model=128
X=tf.keras.random.randint(shape=(1, vocab_size),minval=0, maxval=100)
embed=Embedding(vocab_size, d_model)
embed(X)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:393: UserWarning: `build()` was called on layer 'embedding_2', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


<tf.Tensor: shape=(1, 100, 128), dtype=float32, numpy=
array([[[-0.01246591, -0.07936244,  0.0095226 , ...,  1.0611341 ,
          1.0513012 ,  0.9057236 ],
        [ 0.75755656,  0.79451036,  0.63967824, ...,  1.0810444 ,
          0.9853742 ,  0.93832445],
        [ 0.8496162 ,  0.96761787,  0.93885434, ...,  0.99982446,
          1.0511768 ,  1.0591438 ],
        ...,
        [ 0.43735874,  0.694821  , -0.42196733, ...,  0.9779285 ,
          1.0964279 ,  0.94350916],
        [-0.48622566, -0.05267933, -0.8771938 , ...,  0.9981024 ,
          0.9846013 ,  1.0281209 ],
        [-0.9801623 , -0.87364966, -1.0073472 , ...,  0.9070833 ,
          0.93948674,  1.0234416 ]]], dtype=float32)>

In [21]:
class MultiHead_Attention(tf.keras.layers.Layer):
  def __init__(self, d_model, num_heads, max_seq_len=512):
    super(MultiHead_Attention, self).__init__()
    self.num_heads = num_heads
    self.d_model = d_model
    assert d_model % self.num_heads == 0
    self.head_dim = d_model // self.num_heads
    self.wq = tf.keras.layers.Dense(self.head_dim*self.num_heads, use_bias=False)
    self.wk = tf.keras.layers.Dense(self.head_dim*self.num_heads, use_bias=False)
    self.wv = tf.keras.layers.Dense(self.head_dim*self.num_heads, use_bias=False)
    self.wo=tf.keras.layers.Dense(self.d_model, use_bias=False)

  def call(self, X, Mask=None, Causal=False):
    B,S,D=X.shape
    q=self.wq(X)
    k=self.wk(X)
    v=self.wv(X)

    q=tf.reshape(q, shape=(B,S,self.num_heads, self.head_dim))
    k=tf.reshape(k, shape=(B,S,self.num_heads, self.head_dim))
    v=tf.reshape(v, shape=(B,S, self.num_heads, self.head_dim))
    q=tf.transpose(q, perm=[0,2,1,3])
    k=tf.transpose(k, perm=[0,2,1,3])
    v=tf.transpose(v, perm=[0,2,1,3])

    # Scaled Dot Prod Attention
    scores=tf.matmul(q,k, transpose_b=True)/tf.sqrt(tf.cast(self.head_dim, tf.float32))

    if Mask is not None:
      mask=Mask
      mask=tf.reshape(mask, shape=(B,1,1,S))
      mask=tf.cast(mask, tf.bool)
      scores=scores+((1-mask)*-1e9)
    if Causal is True:
      causal_mask=tf.linalg.band_part(tf.ones(shape=(S,S), dtype=tf.bool),-1,0)
      causal_mask=tf.reshape(causal_mask, shape=(1,1,S,S))
      scores=tf.where(causal_mask, scores, tf.float32.min)

    attn=tf.nn.softmax(scores, axis=-1)
    out=tf.matmul(attn,v)
    out=tf.transpose(out, perm=[0,2,1,3])
    out=tf.reshape(out, shape=(B,S,self.num_heads*self.head_dim))
    out=self.wo(out)
    return out

In [22]:
class FFN(tf.keras.layers.Layer):
  def __init__(self, n_embed, dropout=0.2):
    super(FFN, self).__init__()
    self.n_embed = n_embed
    self.dense1 = tf.keras.layers.Dense(4*n_embed)
    self.dense2 = tf.keras.layers.Dense(n_embed)
    self.dropout = tf.keras.layers.Dropout(dropout)
    self.activation = tf.keras.layers.ReLU()

  def call(self, X):
    X=self.dense1(X)
    X=self.activation(X)
    X=self.dropout(X)
    X=self.dense2(X)
    return X


In [23]:
class Transformer_block(tf.keras.layers.Layer):
  def __init__(self, d_model, num_heads, dropout=0.2):
    super(Transformer_block, self).__init__()
    self.d_model = d_model
    self.num_heads = num_heads
    self.attn = MultiHead_Attention(d_model, num_heads)
    self.ffn = FFN(d_model)
    self.layer_norm1=tf.keras.layers.LayerNormalization()
    self.layer_norm2=tf.keras.layers.LayerNormalization()
    self.dropout=tf.keras.layers.Dropout(dropout)

  def call(self, X, Mask=None, Causal=None):
    X=self.layer_norm1(X)
    attn_out=self.attn(X, Mask=Mask, Causal=Causal)
    X=X+self.dropout(attn_out)
    X=self.layer_norm2(X)
    ffn_out=self.ffn(X)
    X=X+self.dropout(ffn_out)
    return X



In [28]:
class BERT(tf.keras.Model):
  def __init__(self, vocab_size, d_model, num_heads, num_layers, dropout=0.2):
    super(BERT, self).__init__()
    self.vocab_size=vocab_size
    self.d_model=d_model
    self.num_heads=num_heads
    self.num_layers=num_layers
    self.dropout=dropout
    self.Embedding=Embedding(self.vocab_size, self.d_model)
    self.transformer_layers=[Transformer_block(self.d_model, self.num_heads, self.dropout) for _ in range(self.num_layers)]
    self.dropout=tf.keras.layers.Dropout(self.dropout)

  def call(self,x, segment_id=None, mask=None, causal=False, training=False):
    x=self.Embedding(x,segment_id=segment_id, training=training)
    for i in range(self.num_layers):
      x=self.transformer_layers[i](x, Mask=mask, Causal=causal)
    x=self.dropout(x)
    return x

In [29]:
num_heads=8
num_layers=6
vocab_size=100
d_model=128
X=tf.keras.random.randint(shape=(1, vocab_size),minval=0, maxval=100)

In [30]:
model=BERT(vocab_size, d_model, num_heads, num_layers)
model(X)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:393: UserWarning: `build()` was called on layer 'embedding_17', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


<tf.Tensor: shape=(1, 100, 128), dtype=float32, numpy=
array([[[ 0.4647084 , -1.4462656 , -0.3834977 , ..., -0.8404567 ,
          0.2595059 , -1.422707  ],
        [ 0.6669791 , -1.2440552 , -0.24845153, ..., -0.72025055,
          0.34057862, -1.4404811 ],
        [ 0.6655612 , -1.1671757 , -0.25687143, ..., -0.69697493,
          0.3513916 , -1.5106573 ],
        ...,
        [ 0.2542474 , -1.2313721 , -0.6164783 , ..., -0.7772327 ,
          0.3565944 , -1.3997166 ],
        [ 0.15324229, -1.3114505 , -0.6652057 , ..., -0.83749115,
          0.36481827, -1.3914006 ],
        [ 0.14316905, -1.4273212 , -0.6073222 , ..., -0.86524266,
          0.3528983 , -1.4211565 ]]], dtype=float32)>

In [31]:
for layers in model.layers:
  print(layers.name, layers.trainable_variables)

embedding_17 [<Variable path=bert_4/embedding_17/embedding_18/embeddings, shape=(100, 128), dtype=float32, value=[[-0.03469568 -0.03299066 -0.00781959 ...  0.00880529 -0.00252092
  -0.02630954]
 [ 0.01909312  0.03559411 -0.04840446 ...  0.01440806  0.00529654
  -0.03442255]
 [-0.04213011  0.04080597 -0.025975   ... -0.00340527  0.01476283
  -0.00039025]
 ...
 [ 0.01122929 -0.01617633 -0.02198821 ...  0.04850339  0.00060214
  -0.00011893]
 [ 0.03416497  0.03685249 -0.01536378 ...  0.0283044  -0.04032142
  -0.03703132]
 [-0.0459559   0.0135016   0.04653824 ...  0.01574716 -0.03869183
  -0.00478262]]>]
transformer_block_24 [<Variable path=bert_4/transformer_block_24/multi_head__attention_24/dense_144/kernel, shape=(128, 128), dtype=float32, value=[[-0.09403704 -0.09393692 -0.10716504 ... -0.11485039  0.10536878
   0.10571964]
 [-0.01294641  0.09968792  0.12458457 ... -0.0784015   0.07817999
  -0.11515155]
 [-0.0033839   0.14641638  0.1422201  ... -0.03731905  0.14029263
   0.13178314]
 ..

In [32]:
def count_parameters(model):
    """
    Calculate the total number of trainable parameters in a TensorFlow model.

    Args:
        model: A tf.keras.Model or tf.keras.layers.Layer instance.

    Returns:
        int: Total number of trainable parameters.
    """
    total_params = 0
    for variable in model.trainable_variables:
        # Get the shape of the variable (e.g., weights or biases)
        shape = variable.shape
        # Calculate the number of parameters in this variable
        num_params = tf.reduce_prod(shape).numpy()
        total_params += num_params
        # Optional: Print details for each variable
        print(f"Variable: {variable.name}, Shape: {shape}, Parameters: {num_params}")
    return total_params

In [33]:
total_params = count_parameters(model)
print(f"\nTotal trainable parameters: {total_params:,}")

Variable: embeddings, Shape: (100, 128), Parameters: 12800
Variable: kernel, Shape: (128, 128), Parameters: 16384
Variable: kernel, Shape: (128, 128), Parameters: 16384
Variable: kernel, Shape: (128, 128), Parameters: 16384
Variable: kernel, Shape: (128, 128), Parameters: 16384
Variable: kernel, Shape: (128, 512), Parameters: 65536
Variable: bias, Shape: (512,), Parameters: 512
Variable: kernel, Shape: (512, 128), Parameters: 65536
Variable: bias, Shape: (128,), Parameters: 128
Variable: gamma, Shape: (128,), Parameters: 128
Variable: beta, Shape: (128,), Parameters: 128
Variable: gamma, Shape: (128,), Parameters: 128
Variable: beta, Shape: (128,), Parameters: 128
Variable: kernel, Shape: (128, 128), Parameters: 16384
Variable: kernel, Shape: (128, 128), Parameters: 16384
Variable: kernel, Shape: (128, 128), Parameters: 16384
Variable: kernel, Shape: (128, 128), Parameters: 16384
Variable: kernel, Shape: (128, 512), Parameters: 65536
Variable: bias, Shape: (512,), Parameters: 512
Varia